In [0]:
%sql
-- Revenue per order line: only count orders that actually generated revenue
CREATE OR REPLACE TABLE workspace.gold.daily_sales AS
SELECT
  DATE(o.order_ts)                  AS order_date,
  COUNT(DISTINCT o.order_id)        AS orders,
  SUM(o.quantity)                   AS units_sold,
  ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue
FROM workspace.silver.orders o
JOIN workspace.silver.products p ON o.product_id = p.product_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY DATE(o.order_ts);

CREATE OR REPLACE TABLE workspace.gold.product_performance AS
SELECT
  p.product_id, p.product_name, p.category,
  SUM(o.quantity)                          AS units_sold,
  ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue,
  ROUND(AVG(CASE WHEN o.status = 'returned' THEN 1.0 ELSE 0.0 END), 3) AS return_rate
FROM workspace.silver.orders o
JOIN workspace.silver.products p ON o.product_id = p.product_id
WHERE o.status <> 'cancelled'
GROUP BY p.product_id, p.product_name, p.category;

CREATE OR REPLACE TABLE workspace.gold.customer_summary AS
SELECT
  c.customer_id, c.country, c.signup_date,
  COUNT(DISTINCT o.order_id)               AS total_orders,
  ROUND(SUM(o.quantity * p.unit_price), 2) AS lifetime_value,
  MIN(o.order_ts)                          AS first_order,
  MAX(o.order_ts)                          AS last_order
FROM workspace.silver.customers c
JOIN workspace.silver.orders o   ON c.customer_id = o.customer_id
JOIN workspace.silver.products p ON o.product_id = p.product_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY c.customer_id, c.country, c.signup_date;

In [0]:
%sql
SELECT category, ROUND(SUM(revenue),0) AS revenue FROM workspace.gold.product_performance GROUP BY category ORDER BY revenue DESC;
SELECT * FROM workspace.gold.customer_summary ORDER BY lifetime_value DESC LIMIT 10;